# Russian Lip Reading Baseline, Semester 4 - Kaggle

Kaggle version of the first PyTorch baseline on the expanded Russian lip-reading dataset.

Goal: classify a short mouth-crop video clip into one of 10 Russian word classes.


## 1. Kaggle Dataset Setup

This notebook expects the dataset to be attached as a Kaggle Dataset.

In Kaggle, click **Add Input** and add the uploaded lip-reading dataset. Kaggle usually exposes it under `/kaggle/input/<dataset-slug>/`.


In [ ]:
from pathlib import Path
import zipfile
import shutil

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

print("Kaggle input exists:", KAGGLE_INPUT.exists())
if KAGGLE_INPUT.exists():
    print("Available inputs:")
    for item in sorted(KAGGLE_INPUT.iterdir()):
        print("-", item)


## 2. Find Dataset

This cell finds `ru_dataset` inside Kaggle inputs. If the dataset is still stored as a zip file, it extracts it into `/kaggle/working/lipreading_data`.


In [ ]:
def find_ru_dataset():
    # Case 1: Kaggle has already extracted the uploaded zip.
    for candidate in KAGGLE_INPUT.rglob("ru_dataset"):
        if (candidate / "mouth_crops_padded").exists():
            return candidate

    # Case 2: Kaggle input contains the zip as a file.
    zip_names = ["kaggle_lipreading_dataset.zip", "colab_lipreading_dataset.zip"]
    zip_matches = []
    for zip_name in zip_names:
        zip_matches.extend(KAGGLE_INPUT.rglob(zip_name))

    if zip_matches:
        zip_path = zip_matches[0]
        extract_dir = KAGGLE_WORKING / "lipreading_data"
        expected = extract_dir / "ru_dataset" / "mouth_crops_padded"

        if not expected.exists():
            if extract_dir.exists():
                shutil.rmtree(extract_dir)
            extract_dir.mkdir(parents=True, exist_ok=True)

            with zipfile.ZipFile(zip_path, "r") as zip_ref:
                for member in zip_ref.infolist():
                    fixed_name = member.filename.replace("\\", "/")
                    target_path = extract_dir / fixed_name

                    if fixed_name.endswith("/"):
                        target_path.mkdir(parents=True, exist_ok=True)
                        continue

                    target_path.parent.mkdir(parents=True, exist_ok=True)
                    with zip_ref.open(member) as source, open(target_path, "wb") as target:
                        target.write(source.read())

        return extract_dir / "ru_dataset"

    raise FileNotFoundError(
        "Could not find ru_dataset. Add kaggle_lipreading_dataset.zip as a Kaggle Dataset input first."
    )


data_root = find_ru_dataset()
print("data_root:", data_root)
print("mouth crops exist:", (data_root / "mouth_crops_padded").exists())


## 3. Imports, Paths, Tables, Vocabulary

`train_df`, `val_df`, and `test_df` are pandas tables loaded from CSV files.

`vocab` is the list of word classes. `word_to_idx` converts a word into a class number.


In [ ]:
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

split_name = "ml_splits_mouth_crops_padded_clean_manual_no_spk09_10"
split_dir = data_root / split_name / "speaker_top10"

if not (split_dir / "train.csv").exists():
    split_name = "ml_splits_mouth_crops_padded_no_spk09_10"
    split_dir = data_root / split_name / "speaker_top10"

print("split_dir:", split_dir)

train_df = pd.read_csv(split_dir / "train.csv")
val_df = pd.read_csv(split_dir / "val.csv")
test_df = pd.read_csv(split_dir / "test.csv")

vocab = (split_dir / "vocab.txt").read_text(encoding="utf-8").splitlines()
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for word, i in word_to_idx.items()}

device = "cuda" if torch.cuda.is_available() else "cpu"

print("train:", len(train_df))
print("val:", len(val_df))
print("test:", len(test_df))
print("vocab:", vocab)
print("device:", device)
train_df.head()


## 4. Check Class Distribution

This shows that the dataset is imbalanced: some words occur much more often than others.

In [ ]:
print("Train word counts:")
print(train_df["word"].value_counts())

print("\nValidation word counts:")
print(val_df["word"].value_counts())

print("\nTest word counts:")
print(test_df["word"].value_counts())


## 5. Video Loading

`load_video` converts one mp4 clip into a PyTorch tensor.

The default mode is RGB, so the output shape is `[3, frames, height, width]`, for example `[3, 24, 96, 96]`.

For the grayscale experiment we can call `load_video(..., grayscale=True)`. Then the output shape is `[1, frames, height, width]`.



In [ ]:
def load_video(path, num_frames=24, size=96, grayscale=False):
    cap = cv2.VideoCapture(str(path))

    frames = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        raise ValueError(f'Cannot read video: {path}')

    frame_indices = np.linspace(0, total_frames - 1, num_frames).astype(int)

    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()

        if not ok:
            continue

        if grayscale:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            frame = cv2.resize(frame, (size, size))
            frame = frame[:, :, None]
        else:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (size, size))

        frame = frame / 255.0
        frames.append(frame)

    cap.release()

    if len(frames) == 0:
        raise ValueError(f'No frames loaded: {path}')

    while len(frames) < num_frames:
        frames.append(frames[-1])

    video = np.stack(frames)
    video = torch.tensor(video, dtype=torch.float32)
    video = video.permute(3, 0, 1, 2)

    return video



In [ ]:
row = train_df.iloc[0]
video_path = data_root / row['clip_path']

x_rgb = load_video(video_path)
x_gray = load_video(video_path, grayscale=True)

print(video_path)
print('RGB shape:', x_rgb.shape)
print('grayscale shape:', x_gray.shape)
print('RGB min/max:', x_rgb.min().item(), x_rgb.max().item())
print('grayscale min/max:', x_gray.min().item(), x_gray.max().item())
print('word:', row['word'])
print('class id:', word_to_idx[row['word']])



## 6. PyTorch Dataset and DataLoader

Dataset returns one example: (video_tensor, class_id).

DataLoader groups examples into batches.

In [ ]:
class LipReadingDataset(Dataset):
    def __init__(self, df, data_root, word_to_idx, grayscale=False):
        self.df = df.reset_index(drop=True)
        self.data_root = data_root
        self.word_to_idx = word_to_idx
        self.grayscale = grayscale

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        video_path = self.data_root / row['clip_path']
        x = load_video(video_path, grayscale=self.grayscale)

        word = row['word']
        y = self.word_to_idx[word]
        y = torch.tensor(y, dtype=torch.long)

        return x, y



In [ ]:
train_dataset = LipReadingDataset(train_df, data_root, word_to_idx)
val_dataset = LipReadingDataset(val_df, data_root, word_to_idx)
test_dataset = LipReadingDataset(test_df, data_root, word_to_idx)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2)

batch_x, batch_y = next(iter(train_loader))
print('batch_x:', batch_x.shape)
print('batch_y:', batch_y.shape)
print(batch_y)

## 7. Simple 3D-CNN Baseline

Conv3d processes video in time, height, and width.

The model outputs logits with shape [batch_size, num_classes].

In [ ]:
class Simple3DCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv3d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d((1, 2, 2)),

            nn.Conv3d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d((2, 2, 2)),

            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d((2, 2, 2)),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool3d((1, 1, 1)),
            nn.Flatten(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
model = Simple3DCNN(num_classes=len(vocab)).to(device)

batch_x = batch_x.to(device)
batch_y = batch_y.to(device)

outputs = model(batch_x)
print('outputs:', outputs.shape)
print('labels:', batch_y.shape)

## 8. Train and Evaluation Functions

Training updates model weights. Evaluation only measures quality and does not update weights.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)

            total_loss += loss.item() * batch_x.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == batch_y).sum().item()
            total += batch_y.size(0)

    return total_loss / total, correct / total

## 9. Baseline Training

This trains the simple 3D-CNN for a few epochs and saves the best checkpoint by validation accuracy into `/kaggle/working/lipreading_sem4`.


In [ ]:
checkpoint_dir = Path('/kaggle/working/lipreading_sem4')
checkpoint_dir.mkdir(parents=True, exist_ok=True)

model = Simple3DCNN(num_classes=len(vocab)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 5
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_checkpoint_path = checkpoint_dir / 'simple3dcnn_best.pt'

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(
        f'epoch {epoch + 1}/{num_epochs} | '
        f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} | '
        f'val_loss={val_loss:.4f} val_acc={val_acc:.4f}'
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(
            {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'vocab': vocab,
                'word_to_idx': word_to_idx,
                'idx_to_word': idx_to_word,
                'history': history,
                'best_val_acc': best_val_acc,
            },
            best_checkpoint_path,
        )
        print('saved best:', best_checkpoint_path)

## 10. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='train')
plt.plot(history['val_loss'], label='val')
plt.title('Loss')
plt.xlabel('epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='train')
plt.plot(history['val_acc'], label='val')
plt.title('Accuracy')
plt.xlabel('epoch')
plt.legend()

plt.tight_layout()
plt.show()

## 11. Prediction Diagnostics

These functions check whether the model predicts many classes or collapses into one frequent class.

They show:

- prediction distribution;
- true label distribution;
- per-class accuracy;
- confusion matrix.

In [ ]:
def collect_predictions(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)
            preds = outputs.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(batch_y.cpu().numpy().tolist())

    return all_preds, all_labels


def prediction_report(name, preds, labels, idx_to_word):
    pred_counts = Counter(preds)
    label_counts = Counter(labels)
    correct = sum(int(pred == label) for pred, label in zip(preds, labels))
    total = len(labels)
    accuracy = correct / total if total else 0.0

    print(name)
    print('accuracy:', round(accuracy, 4))

    print('\npredicted words:')
    for idx, count in pred_counts.most_common():
        print(idx_to_word[idx], count)

    print('\ntrue words:')
    for idx, count in label_counts.most_common():
        print(idx_to_word[idx], count)

    rows = []
    for idx in sorted(idx_to_word):
        word = idx_to_word[idx]
        n_true = label_counts[idx]
        n_correct = sum(
            int(pred == label == idx)
            for pred, label in zip(preds, labels)
        )
        class_acc = n_correct / n_true if n_true else 0.0
        rows.append({
            'word': word,
            'true_count': n_true,
            'pred_count': pred_counts[idx],
            'correct': n_correct,
            'class_accuracy': class_acc,
        })

    report_df = pd.DataFrame(rows)
    display(report_df)
    return report_df


def confusion_matrix_array(preds, labels, num_classes):
    matrix = np.zeros((num_classes, num_classes), dtype=int)
    for true_idx, pred_idx in zip(labels, preds):
        matrix[true_idx, pred_idx] += 1
    return matrix


def plot_confusion_matrix(preds, labels, idx_to_word, title):
    words = [idx_to_word[i] for i in range(len(idx_to_word))]
    matrix = confusion_matrix_array(preds, labels, len(words))

    plt.figure(figsize=(8, 7))
    plt.imshow(matrix, cmap='Blues')
    plt.title(title)
    plt.xlabel('predicted word')
    plt.ylabel('true word')
    plt.xticks(range(len(words)), words, rotation=45, ha='right')
    plt.yticks(range(len(words)), words)
    plt.colorbar()

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix[i, j]
            if value:
                plt.text(j, i, str(value), ha='center', va='center')

    plt.tight_layout()
    plt.show()
    return matrix


## 12. Baseline Diagnostics

This checks the plain Simple3DCNN after training with regular CrossEntropyLoss.

In [ ]:
baseline_val_preds, baseline_val_labels = collect_predictions(model, val_loader, device)

baseline_val_report = prediction_report(
    'Simple3DCNN + CrossEntropyLoss on validation',
    baseline_val_preds,
    baseline_val_labels,
    idx_to_word,
)

baseline_confusion = plot_confusion_matrix(
    baseline_val_preds,
    baseline_val_labels,
    idx_to_word,
    'Simple3DCNN + CrossEntropyLoss: validation confusion matrix',
)


## 13. Weighted Loss Experiment

Class weights increase the loss penalty for rare classes and reduce the penalty for frequent classes.

This changes the loss function, but the training batches are still sampled from the original imbalanced dataset.

In [ ]:
train_counts = Counter(train_df['word'])
print(train_counts)

class_weights = []
for word in vocab:
    count = train_counts[word]
    class_weights.append(1.0 / count)

class_weights = torch.tensor(class_weights, dtype=torch.float32)
class_weights = class_weights / class_weights.mean()
class_weights = class_weights.to(device)

print(class_weights)


In [ ]:
weighted_model = Simple3DCNN(num_classes=len(vocab)).to(device)
weighted_criterion = nn.CrossEntropyLoss(weight=class_weights)
weighted_optimizer = torch.optim.Adam(weighted_model.parameters(), lr=1e-3)

num_epochs = 5
history_weighted = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_checkpoint_path = checkpoint_dir / 'simple3dcnn_weighted_best.pt'

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(weighted_model, train_loader, weighted_criterion, weighted_optimizer, device)
    val_loss, val_acc = evaluate(weighted_model, val_loader, weighted_criterion, device)

    history_weighted['train_loss'].append(train_loss)
    history_weighted['train_acc'].append(train_acc)
    history_weighted['val_loss'].append(val_loss)
    history_weighted['val_acc'].append(val_acc)

    print(
        f'epoch {epoch + 1}/{num_epochs} | '
        f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} | '
        f'val_loss={val_loss:.4f} val_acc={val_acc:.4f}'
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(
            {
                'epoch': epoch + 1,
                'model_state_dict': weighted_model.state_dict(),
                'optimizer_state_dict': weighted_optimizer.state_dict(),
                'vocab': vocab,
                'history': history_weighted,
                'best_val_acc': best_val_acc,
                'class_weights': class_weights.detach().cpu(),
            },
            best_checkpoint_path,
        )
        print('saved best:', best_checkpoint_path)


## 14. Weighted Loss Diagnostics

This checks whether weighted loss made the model predict more than one class.

In [ ]:
weighted_val_preds, weighted_val_labels = collect_predictions(weighted_model, val_loader, device)

weighted_val_report = prediction_report(
    'Simple3DCNN + weighted CrossEntropyLoss on validation',
    weighted_val_preds,
    weighted_val_labels,
    idx_to_word,
)

weighted_confusion = plot_confusion_matrix(
    weighted_val_preds,
    weighted_val_labels,
    idx_to_word,
    'Simple3DCNN + weighted CrossEntropyLoss: validation confusion matrix',
)


## 15. Balanced Sampler Experiment

`WeightedRandomSampler` changes how training batches are formed.

Instead of only changing the loss value, it samples clips from rare classes more often, so batches become more balanced.

In [ ]:
from torch.utils.data import WeightedRandomSampler

train_label_ids = [word_to_idx[word] for word in train_df['word']]
label_counts = Counter(train_label_ids)

sample_weights = torch.tensor(
    [1.0 / label_counts[label_id] for label_id in train_label_ids],
    dtype=torch.float32,
)

balanced_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

balanced_train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    sampler=balanced_sampler,
    shuffle=False,
    num_workers=2,
)

batch_x_balanced, batch_y_balanced = next(iter(balanced_train_loader))
print('balanced batch labels:', batch_y_balanced)
print('balanced batch words:')
print([idx_to_word[idx.item()] for idx in batch_y_balanced])


In [ ]:
balanced_model = Simple3DCNN(num_classes=len(vocab)).to(device)
balanced_criterion = nn.CrossEntropyLoss()
balanced_optimizer = torch.optim.Adam(balanced_model.parameters(), lr=1e-3)

num_epochs = 5
history_balanced = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_checkpoint_path = checkpoint_dir / 'simple3dcnn_balanced_sampler_best.pt'

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(balanced_model, balanced_train_loader, balanced_criterion, balanced_optimizer, device)
    val_loss, val_acc = evaluate(balanced_model, val_loader, balanced_criterion, device)

    history_balanced['train_loss'].append(train_loss)
    history_balanced['train_acc'].append(train_acc)
    history_balanced['val_loss'].append(val_loss)
    history_balanced['val_acc'].append(val_acc)

    print(
        f'epoch {epoch + 1}/{num_epochs} | '
        f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} | '
        f'val_loss={val_loss:.4f} val_acc={val_acc:.4f}'
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(
            {
                'epoch': epoch + 1,
                'model_state_dict': balanced_model.state_dict(),
                'optimizer_state_dict': balanced_optimizer.state_dict(),
                'vocab': vocab,
                'history': history_balanced,
                'best_val_acc': best_val_acc,
            },
            best_checkpoint_path,
        )
        print('saved best:', best_checkpoint_path)


## 16. Balanced Sampler Diagnostics

This checks whether balanced sampling improves prediction diversity and per-class accuracy.

In [ ]:
balanced_val_preds, balanced_val_labels = collect_predictions(balanced_model, val_loader, device)

balanced_val_report = prediction_report(
    'Simple3DCNN + WeightedRandomSampler on validation',
    balanced_val_preds,
    balanced_val_labels,
    idx_to_word,
)

balanced_confusion = plot_confusion_matrix(
    balanced_val_preds,
    balanced_val_labels,
    idx_to_word,
    'Simple3DCNN + WeightedRandomSampler: validation confusion matrix',
)


## 17. Compare Simple3DCNN Experiments

This table compares the best validation accuracy from the three baseline experiments.

Important: accuracy alone can be misleading, so also check the prediction reports and confusion matrices above.


In [ ]:
summary = pd.DataFrame([
    {
        'experiment': 'Simple3DCNN + CrossEntropyLoss',
        'best_val_acc': max(history['val_acc']),
        'last_val_acc': history['val_acc'][-1],
    },
    {
        'experiment': 'Simple3DCNN + weighted CrossEntropyLoss',
        'best_val_acc': max(history_weighted['val_acc']),
        'last_val_acc': history_weighted['val_acc'][-1],
    },
    {
        'experiment': 'Simple3DCNN + WeightedRandomSampler',
        'best_val_acc': max(history_balanced['val_acc']),
        'last_val_acc': history_balanced['val_acc'][-1],
    },
])

summary


## 18. Why Add a Temporal Model

The simple 3D-CNN baseline is useful as a first check, but it collapsed to predicting frequent classes.

For lip reading, the order of mouth movements matters. A stronger baseline should therefore have two parts:

1. A frame encoder that extracts visual features from each mouth frame.
2. A temporal encoder that reads the sequence of frame features.

Here we use a small CNN as the frame encoder and a bidirectional GRU as the temporal encoder.



## 19. RGB and Grayscale Loaders

This creates separate DataLoaders for RGB and grayscale experiments.

RGB input shape: `[batch, 3, frames, height, width]`.

Grayscale input shape: `[batch, 1, frames, height, width]`.



In [ ]:
from torch.utils.data import WeightedRandomSampler


def make_loaders(grayscale=False, batch_size=8, balanced=False):
    train_dataset = LipReadingDataset(train_df, data_root, word_to_idx, grayscale=grayscale)
    val_dataset = LipReadingDataset(val_df, data_root, word_to_idx, grayscale=grayscale)
    test_dataset = LipReadingDataset(test_df, data_root, word_to_idx, grayscale=grayscale)

    if balanced:
        class_counts = Counter(train_df['word'])
        sample_weights = [1.0 / class_counts[word] for word in train_df['word']]
        sampler = WeightedRandomSampler(
            weights=torch.tensor(sample_weights, dtype=torch.float32),
            num_samples=len(sample_weights),
            replacement=True,
        )
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            sampler=sampler,
            num_workers=2,
        )
    else:
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=2,
        )

    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    return train_loader, val_loader, test_loader


rgb_train_loader, rgb_val_loader, rgb_test_loader = make_loaders(grayscale=False, batch_size=8)
gray_train_loader, gray_val_loader, gray_test_loader = make_loaders(grayscale=True, batch_size=8)

gray_batch_x, gray_batch_y = next(iter(gray_train_loader))
print('gray batch_x:', gray_batch_x.shape)
print('gray batch_y:', gray_batch_y.shape)
print(gray_batch_y)



## 20. Frame CNN + BiGRU Model

This model processes a clip as a sequence.

Input:

```text
[B, C, T, H, W]
```

Steps:

```text
video
-> split into frames
-> CNN extracts one feature vector per frame
-> BiGRU reads the feature sequence in both time directions
-> Linear classifier predicts the word
```



In [ ]:
class FrameCNNBiGRU(nn.Module):
    def __init__(self, num_classes, in_channels=3, cnn_dim=64, gru_hidden=64, dropout=0.3):
        super().__init__()

        self.frame_encoder = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, cnn_dim, kernel_size=3, padding=1),
            nn.BatchNorm2d(cnn_dim),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
        )

        self.temporal_encoder = nn.GRU(
            input_size=cnn_dim,
            hidden_size=gru_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(gru_hidden * 2, num_classes),
        )

    def forward(self, x):
        # x: [B, C, T, H, W]
        batch_size, channels, frames, height, width = x.shape

        # Move time next to batch: [B, T, C, H, W]
        x = x.permute(0, 2, 1, 3, 4)

        # Encode every frame with the same CNN: [B*T, C, H, W] -> [B*T, cnn_dim]
        x = x.reshape(batch_size * frames, channels, height, width)
        frame_features = self.frame_encoder(x)

        # Restore the time axis: [B, T, cnn_dim]
        frame_features = frame_features.reshape(batch_size, frames, -1)

        _, hidden = self.temporal_encoder(frame_features)

        # For a bidirectional GRU, the last two hidden states are forward and backward.
        forward_last = hidden[-2]
        backward_last = hidden[-1]
        clip_features = torch.cat([forward_last, backward_last], dim=1)

        logits = self.classifier(clip_features)
        return logits


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


rgb_bigru_model = FrameCNNBiGRU(num_classes=len(vocab), in_channels=3).to(device)
gray_bigru_model = FrameCNNBiGRU(num_classes=len(vocab), in_channels=1).to(device)

print('RGB CNN+BiGRU parameters:', count_parameters(rgb_bigru_model))
print('grayscale CNN+BiGRU parameters:', count_parameters(gray_bigru_model))

with torch.no_grad():
    test_logits = gray_bigru_model(gray_batch_x.to(device))

print('test logits:', test_logits.shape)



## 21. Reusable Training Helper

This is the same training logic as before, wrapped into one function so that experiments are easier to compare.



In [ ]:
def fit_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs, checkpoint_path, experiment_name):
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(
            f'{experiment_name} | epoch {epoch + 1}/{num_epochs} | '
            f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} | '
            f'val_loss={val_loss:.4f} val_acc={val_acc:.4f}'
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(
                {
                    'epoch': epoch + 1,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'vocab': vocab,
                    'word_to_idx': word_to_idx,
                    'idx_to_word': idx_to_word,
                    'history': history,
                    'best_val_acc': best_val_acc,
                    'experiment_name': experiment_name,
                },
                checkpoint_path,
            )
            print('saved best:', checkpoint_path)

    return history, best_val_acc



## 22. Train CNN+BiGRU on RGB Clips

This is the stronger temporal baseline with RGB input.



In [ ]:
rgb_bigru_model = FrameCNNBiGRU(num_classes=len(vocab), in_channels=3).to(device)
rgb_bigru_criterion = nn.CrossEntropyLoss()
rgb_bigru_optimizer = torch.optim.AdamW(rgb_bigru_model.parameters(), lr=1e-3, weight_decay=1e-4)
rgb_bigru_checkpoint_path = checkpoint_dir / 'framecnn_bigru_rgb_best.pt'

rgb_bigru_history, rgb_bigru_best_val_acc = fit_model(
    model=rgb_bigru_model,
    train_loader=rgb_train_loader,
    val_loader=rgb_val_loader,
    criterion=rgb_bigru_criterion,
    optimizer=rgb_bigru_optimizer,
    device=device,
    num_epochs=8,
    checkpoint_path=rgb_bigru_checkpoint_path,
    experiment_name='FrameCNN+BiGRU RGB',
)

rgb_bigru_checkpoint = torch.load(rgb_bigru_checkpoint_path, map_location=device)
rgb_bigru_model.load_state_dict(rgb_bigru_checkpoint['model_state_dict'])
print('loaded best RGB checkpoint:', rgb_bigru_checkpoint_path)
print('best RGB val_acc:', rgb_bigru_checkpoint['best_val_acc'])



## 23. RGB CNN+BiGRU Diagnostics

Again, accuracy alone is not enough. We check whether the model predicts several classes or collapses to one word.



In [ ]:
rgb_bigru_val_preds, rgb_bigru_val_labels = collect_predictions(rgb_bigru_model, rgb_val_loader, device)

rgb_bigru_report = prediction_report(
    'FrameCNN+BiGRU RGB on validation',
    rgb_bigru_val_preds,
    rgb_bigru_val_labels,
    idx_to_word,
)

plot_confusion_matrix(
    rgb_bigru_val_preds,
    rgb_bigru_val_labels,
    idx_to_word,
    'FrameCNN+BiGRU RGB: validation confusion matrix',
)



## 24. Train CNN+BiGRU on Grayscale Clips

This checks whether grayscale is a useful optimization. If quality is similar or better than RGB, grayscale is preferable because the model receives less irrelevant color information.



In [ ]:
gray_bigru_model = FrameCNNBiGRU(num_classes=len(vocab), in_channels=1).to(device)
gray_bigru_criterion = nn.CrossEntropyLoss()
gray_bigru_optimizer = torch.optim.AdamW(gray_bigru_model.parameters(), lr=1e-3, weight_decay=1e-4)
gray_bigru_checkpoint_path = checkpoint_dir / 'framecnn_bigru_gray_best.pt'

gray_bigru_history, gray_bigru_best_val_acc = fit_model(
    model=gray_bigru_model,
    train_loader=gray_train_loader,
    val_loader=gray_val_loader,
    criterion=gray_bigru_criterion,
    optimizer=gray_bigru_optimizer,
    device=device,
    num_epochs=8,
    checkpoint_path=gray_bigru_checkpoint_path,
    experiment_name='FrameCNN+BiGRU grayscale',
)

gray_bigru_checkpoint = torch.load(gray_bigru_checkpoint_path, map_location=device)
gray_bigru_model.load_state_dict(gray_bigru_checkpoint['model_state_dict'])
print('loaded best grayscale checkpoint:', gray_bigru_checkpoint_path)
print('best grayscale val_acc:', gray_bigru_checkpoint['best_val_acc'])



## 25. Grayscale CNN+BiGRU Diagnostics

This shows whether grayscale reduces collapse to frequent words or loses useful information.



In [ ]:
gray_bigru_val_preds, gray_bigru_val_labels = collect_predictions(gray_bigru_model, gray_val_loader, device)

gray_bigru_report = prediction_report(
    'FrameCNN+BiGRU grayscale on validation',
    gray_bigru_val_preds,
    gray_bigru_val_labels,
    idx_to_word,
)

plot_confusion_matrix(
    gray_bigru_val_preds,
    gray_bigru_val_labels,
    idx_to_word,
    'FrameCNN+BiGRU grayscale: validation confusion matrix',
)



## 26. Compare All Experiments

This table is the main checkpoint for deciding what to do next: improve the model more, clean crops, or expand the dataset.



In [ ]:
all_experiments_summary = pd.DataFrame([
    {
        'experiment': 'Simple3DCNN + CrossEntropyLoss',
        'best_val_acc': max(history['val_acc']),
        'last_val_acc': history['val_acc'][-1],
    },
    {
        'experiment': 'Simple3DCNN + weighted CrossEntropyLoss',
        'best_val_acc': max(history_weighted['val_acc']),
        'last_val_acc': history_weighted['val_acc'][-1],
    },
    {
        'experiment': 'Simple3DCNN + WeightedRandomSampler',
        'best_val_acc': max(history_balanced['val_acc']),
        'last_val_acc': history_balanced['val_acc'][-1],
    },
    {
        'experiment': 'FrameCNN+BiGRU RGB',
        'best_val_acc': max(rgb_bigru_history['val_acc']),
        'last_val_acc': rgb_bigru_history['val_acc'][-1],
    },
    {
        'experiment': 'FrameCNN+BiGRU grayscale',
        'best_val_acc': max(gray_bigru_history['val_acc']),
        'last_val_acc': gray_bigru_history['val_acc'][-1],
    },
])

all_experiments_summary



## 27. Random Split Sanity Check

The speaker-based split is the honest evaluation setup, because validation and test speakers are unseen during training.

But it is also much harder. This sanity check uses the existing `random_top10` split to answer one question:

```text
Can the model learn the visual word classes at all when speakers are mixed across train/validation/test?
```

If random split works much better than speaker split, the main problem is speaker generalization. If it also fails, the problem is likely dataset quality, alignment/crops, or model capacity.



In [ ]:
def find_split_dir(split_variant):
    candidates = [
        data_root / split_name / split_variant,
        data_root / 'ml_splits_mouth_crops_padded_no_spk09_10' / split_variant,
    ]

    for candidate in candidates:
        if (candidate / 'train.csv').exists():
            return candidate

    raise FileNotFoundError(f'Could not find split: {split_variant}')


random_split_dir = find_split_dir('random_top10')
print('random split:', random_split_dir)

random_train_df = pd.read_csv(random_split_dir / 'train.csv')
random_val_df = pd.read_csv(random_split_dir / 'val.csv')
random_test_df = pd.read_csv(random_split_dir / 'test.csv')

random_vocab_path = random_split_dir / 'vocab.txt'
if random_vocab_path.exists():
    random_vocab = [line.strip() for line in random_vocab_path.read_text(encoding='utf-8').splitlines() if line.strip()]
else:
    random_vocab = sorted(random_train_df['word'].unique())

random_word_to_idx = {word: idx for idx, word in enumerate(random_vocab)}
random_idx_to_word = {idx: word for word, idx in random_word_to_idx.items()}

print('random train:', len(random_train_df))
print('random val:', len(random_val_df))
print('random test:', len(random_test_df))
print('random vocab:', random_vocab)
print('random train speakers:', sorted(random_train_df['speaker_id'].unique()))
print('random val speakers:', sorted(random_val_df['speaker_id'].unique()))
print('random test speakers:', sorted(random_test_df['speaker_id'].unique()))

print('\nRandom train word counts:')
print(random_train_df['word'].value_counts())
print('\nRandom validation word counts:')
print(random_val_df['word'].value_counts())




## 28. Train Grayscale CNN+BiGRU on Random Split

We use grayscale here because it was at least as good as RGB on the speaker-based split and is cheaper to train.



In [ ]:
random_gray_train_dataset = LipReadingDataset(random_train_df, data_root, random_word_to_idx, grayscale=True)
random_gray_val_dataset = LipReadingDataset(random_val_df, data_root, random_word_to_idx, grayscale=True)
random_gray_test_dataset = LipReadingDataset(random_test_df, data_root, random_word_to_idx, grayscale=True)

random_gray_train_loader = DataLoader(random_gray_train_dataset, batch_size=8, shuffle=True, num_workers=2)
random_gray_val_loader = DataLoader(random_gray_val_dataset, batch_size=8, shuffle=False, num_workers=2)
random_gray_test_loader = DataLoader(random_gray_test_dataset, batch_size=8, shuffle=False, num_workers=2)

random_gray_model = FrameCNNBiGRU(num_classes=len(random_vocab), in_channels=1).to(device)
random_gray_criterion = nn.CrossEntropyLoss()
random_gray_optimizer = torch.optim.AdamW(random_gray_model.parameters(), lr=1e-3, weight_decay=1e-4)
random_gray_checkpoint_path = checkpoint_dir / 'framecnn_bigru_gray_random_split_best.pt'

random_gray_history, random_gray_best_val_acc = fit_model(
    model=random_gray_model,
    train_loader=random_gray_train_loader,
    val_loader=random_gray_val_loader,
    criterion=random_gray_criterion,
    optimizer=random_gray_optimizer,
    device=device,
    num_epochs=8,
    checkpoint_path=random_gray_checkpoint_path,
    experiment_name='FrameCNN+BiGRU grayscale random split',
)

random_gray_checkpoint = torch.load(random_gray_checkpoint_path, map_location=device)
random_gray_model.load_state_dict(random_gray_checkpoint['model_state_dict'])
print('loaded best random split checkpoint:', random_gray_checkpoint_path)
print('best random split val_acc:', random_gray_checkpoint['best_val_acc'])



## 29. Random Split Diagnostics

If this block looks much better than the speaker-based diagnostics, the model is learning speaker-specific visual patterns and does not yet generalize well to unseen speakers.



In [ ]:
random_gray_val_preds, random_gray_val_labels = collect_predictions(random_gray_model, random_gray_val_loader, device)

random_gray_report = prediction_report(
    'FrameCNN+BiGRU grayscale on random validation split',
    random_gray_val_preds,
    random_gray_val_labels,
    random_idx_to_word,
)

plot_confusion_matrix(
    random_gray_val_preds,
    random_gray_val_labels,
    random_idx_to_word,
    'FrameCNN+BiGRU grayscale random split: validation confusion matrix',
)



## 30. Final Comparison With Sanity Check

Use the speaker-based rows for honest quality. Use the random-split row only as a diagnostic sanity check.



In [ ]:
final_experiments_summary = pd.DataFrame([
    {
        'experiment': 'Simple3DCNN + CrossEntropyLoss, speaker split',
        'best_val_acc': max(history['val_acc']),
        'last_val_acc': history['val_acc'][-1],
        'evaluation_type': 'speaker-based',
    },
    {
        'experiment': 'Simple3DCNN + weighted CrossEntropyLoss, speaker split',
        'best_val_acc': max(history_weighted['val_acc']),
        'last_val_acc': history_weighted['val_acc'][-1],
        'evaluation_type': 'speaker-based',
    },
    {
        'experiment': 'Simple3DCNN + WeightedRandomSampler, speaker split',
        'best_val_acc': max(history_balanced['val_acc']),
        'last_val_acc': history_balanced['val_acc'][-1],
        'evaluation_type': 'speaker-based',
    },
    {
        'experiment': 'FrameCNN+BiGRU RGB, speaker split',
        'best_val_acc': max(rgb_bigru_history['val_acc']),
        'last_val_acc': rgb_bigru_history['val_acc'][-1],
        'evaluation_type': 'speaker-based',
    },
    {
        'experiment': 'FrameCNN+BiGRU grayscale, speaker split',
        'best_val_acc': max(gray_bigru_history['val_acc']),
        'last_val_acc': gray_bigru_history['val_acc'][-1],
        'evaluation_type': 'speaker-based',
    },
    {
        'experiment': 'FrameCNN+BiGRU grayscale, random split',
        'best_val_acc': max(random_gray_history['val_acc']),
        'last_val_acc': random_gray_history['val_acc'][-1],
        'evaluation_type': 'random sanity check',
    },
])

final_experiments_summary



## 31. Next Experiments

Recommended next steps after the random split sanity check:

1. If random split is much better than speaker split, add more clean speakers and use stronger regularization/augmentation.
2. If random split is also weak, inspect alignment and mouth crops before adding a more complex model.
3. If grayscale is close to RGB, keep grayscale for future experiments.
4. Do not manually clean everything. First use model errors and high-loss examples to find suspicious clips.
5. Optional stronger model after this sanity check: CNN+TCN.

